# Quadtree Tile Index

Reads image pair labels, selects the coarsest TIFF pyramid page that still covers the CNN input dimensions at each quadtree depth, and writes one entry per `(pair, depth, x, y)` tile to `quadtree_annotations.json`.

In [44]:
from pathlib import Path
import json
import sys

import numpy as np
import tifffile
from PIL import Image

sys.path.append(str(Path.cwd().parent))

import conf
from pair_mask import (
    PRODUCTION_MIN_INSIDE_FRACTION,
    is_tile_excluded_by_polygons,
    load_pair_mask,
    mask_json_path,
    scaled_polygons_for_page,
)


TEST_RUN = False
TEST_RUN_TILE_LIMIT = 100
LOG_EVERY = 300

IMAGE_DIR = conf.IMAGE_DIR
LABELS_PATH = conf.LABELS_PATH
ANNOTATION_PATH = conf.ANNOTATION_PATH

WSI_PAGES = conf.WSI_PAGES
CNN_INPUT_HEIGHT = conf.CNN_INPUT_HEIGHT
CNN_INPUT_WIDTH = conf.CNN_INPUT_WIDTH
MAX_CROP_DEPTH = conf.MAX_CROP_DEPTH

print(f"System   : {conf.SYSTEM_PREFIX}")
print(f"Test run : {TEST_RUN}")
print(f"Labels   : {LABELS_PATH}")
print(f"Output   : {ANNOTATION_PATH}")

System   : macos
Test run : False
Labels   : /Users/alexanderhallmann/Desktop/medical-image-registration/data/macos_labels.json
Output   : /Users/alexanderhallmann/Desktop/medical-image-registration/data/macos_quadtree_annotations.json


In [45]:
from setup.background_filter import is_background_tile

In [46]:
def matrix_dict_to_list(m):
    return [
        [m["t_00"], m["t_01"], m["t_02"]],
        [m["t_10"], m["t_11"], m["t_12"]],
        [m["t_20"], m["t_21"], m["t_22"]],
    ]


def load_pairs_from_labels(labels_path, image_dir):
    with open(labels_path, "r", encoding="utf-8") as f:
        labels = json.load(f)

    pairs = []

    for pair_id, item in enumerate(labels):
        source_id = item["source_image_id"]
        target_id = item["target_image_id"]

        source_path = image_dir / f"{source_id}.data"
        target_path = image_dir / f"{target_id}.data"

        if not source_path.exists():
            raise FileNotFoundError(source_path)

        if not target_path.exists():
            raise FileNotFoundError(target_path)

        pairs.append({
            "pair_id": pair_id,
            "moving_path": conf.image_relpath(source_id),
            "fixed_path": conf.image_relpath(target_id),
            "source_image_id": source_id,
            "target_image_id": target_id,
            "registration_error": item["registration_error"],
            "transformation_matrix": matrix_dict_to_list(item["transformation_matrix"]),
        })

    return pairs

In [47]:
def choose_pair_pyramid_page(fixed_path, moving_path, crop_depth, input_height=CNN_INPUT_HEIGHT, input_width=CNN_INPUT_WIDTH):
    fixed_path = conf.resolve(fixed_path)
    moving_path = conf.resolve(moving_path)
    grid = 2 ** crop_depth

    with tifffile.TiffFile(fixed_path) as fixed_slide, tifffile.TiffFile(moving_path) as moving_slide:
        candidates = []

        for page_idx in WSI_PAGES:
            fixed_h, fixed_w = fixed_slide.pages[page_idx].shape[:2]
            moving_h, moving_w = moving_slide.pages[page_idx].shape[:2]

            fixed_tile_h = fixed_h // grid
            fixed_tile_w = fixed_w // grid

            moving_tile_h = moving_h // grid
            moving_tile_w = moving_w // grid

            min_tile_h = min(fixed_tile_h, moving_tile_h)
            min_tile_w = min(fixed_tile_w, moving_tile_w)

            if min_tile_h >= input_height and min_tile_w >= input_width:
                candidates.append((page_idx, min_tile_h, min_tile_w))

    if not candidates:
        return None

    return candidates[-1]

In [48]:
def page_shape(path, pyramid_page_idx, shape_cache):
    path = conf.resolve(path)
    key = (str(path), pyramid_page_idx)
    if key not in shape_cache:
        with tifffile.TiffFile(path) as slide:
            shape_cache[key] = slide.pages[pyramid_page_idx].shape[:2]
    return shape_cache[key]


def he_tile_excluded(pair, job_fields, shape_cache, polygon_cache):
    pyramid_page_idx = job_fields["pyramid_page_idx"]
    page_h, page_w = page_shape(pair["fixed_path"], pyramid_page_idx, shape_cache)
    poly_key = (pair["pair_id"], page_w, page_h)
    if poly_key not in polygon_cache:
        polygons = scaled_polygons_for_page(pair["pair_id"], page_w, page_h)
        if polygons is None:
            raise FileNotFoundError(
                f"No HE mask for pair {pair['pair_id']}: {mask_json_path(pair['pair_id'])}"
            )
        polygon_cache[poly_key] = polygons
    return is_tile_excluded_by_polygons(
        polygon_cache[poly_key],
        job_fields["grid"],
        job_fields["x_idx"],
        job_fields["y_idx"],
        page_h,
        page_w,
        PRODUCTION_MIN_INSIDE_FRACTION,
    )


def build_quadtree_index(
    pairs,
    max_crop_depth,
    input_height=CNN_INPUT_HEIGHT,
    input_width=CNN_INPUT_WIDTH,
    max_tiles=None,
    log_every=LOG_EVERY,
    save_path=None,
):
    tile_jobs = []
    shape_cache = {}
    polygon_cache = {}

    try:
        for pair in pairs:
            print(
                f"pair_id {pair['pair_id']}: "
                f"HE {pair['target_image_id']} / IHC {pair['source_image_id']}"
            )
            for crop_depth in range(max_crop_depth + 1):
                chosen = choose_pair_pyramid_page(
                    fixed_path=pair["fixed_path"],
                    moving_path=pair["moving_path"],
                    crop_depth=crop_depth,
                    input_height=input_height,
                    input_width=input_width,
                )

                if chosen is None:
                    continue

                pyramid_page_idx, tile_h, tile_w = chosen
                grid = 2 ** crop_depth
                print(
                    f"  d{crop_depth}: grid {grid}x{grid}, "
                    f"page {pyramid_page_idx}, tile {tile_w}x{tile_h}"
                )

                for y_idx in range(grid):
                    for x_idx in range(grid):
                        job = {
                            "pair_id": pair["pair_id"],
                            "fixed_path": pair["fixed_path"],
                            "moving_path": pair["moving_path"],
                            "source_image_id": pair["source_image_id"],
                            "target_image_id": pair["target_image_id"],
                            "crop_depth": crop_depth,
                            "grid": grid,
                            "x_idx": x_idx,
                            "y_idx": y_idx,
                            "pyramid_page_idx": pyramid_page_idx,
                            "tile_h": tile_h,
                            "tile_w": tile_w,
                            "cnn_input_height": input_height,
                            "cnn_input_width": input_width,
                            "registration_error": pair["registration_error"],
                            "transformation_matrix": pair["transformation_matrix"],
                        }
                        job["binary_mask_excluded"] = he_tile_excluded(
                            pair, job, shape_cache, polygon_cache,
                        )
                        tile_jobs.append(job)
                        if log_every and len(tile_jobs) % log_every == 0:
                            print(
                                f"  [{len(tile_jobs)} tiles] "
                                f"pair_id={pair['pair_id']} d{crop_depth} ({x_idx},{y_idx})"
                            )
                        if max_tiles is not None and len(tile_jobs) >= max_tiles:
                            return tile_jobs

            print(f"  pair_id {pair['pair_id']} done — {len(tile_jobs)} tiles total so far")

    except KeyboardInterrupt:
        if save_path is not None and tile_jobs:
            save_json(tile_jobs, save_path)
            print(
                f"\nKeyboardInterrupt — saved partial index "
                f"({len(tile_jobs)} jobs) to {save_path}"
            )
        raise

    return tile_jobs


def save_json(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

In [ ]:
pairs = load_pairs_from_labels(LABELS_PATH, IMAGE_DIR)

build_kwargs = dict(
    max_crop_depth=MAX_CROP_DEPTH,
    input_height=CNN_INPUT_HEIGHT,
    input_width=CNN_INPUT_WIDTH,
    save_path=ANNOTATION_PATH,
)

if TEST_RUN:
    first_pair_id = min(pair["pair_id"] for pair in pairs)
    pairs = [pair for pair in pairs if pair["pair_id"] == first_pair_id]
    if load_pair_mask(first_pair_id) is None:
        raise FileNotFoundError(
            f"No HE mask for pair {first_pair_id}: {mask_json_path(first_pair_id)}"
        )
    print(
        f"TEST_RUN: pair_id {first_pair_id}, "
        f"limit {TEST_RUN_TILE_LIMIT} tiles, "
        f"threshold {PRODUCTION_MIN_INSIDE_FRACTION}"
    )
    build_kwargs["max_tiles"] = TEST_RUN_TILE_LIMIT

try:
    tile_jobs = build_quadtree_index(pairs=pairs, **build_kwargs)
except KeyboardInterrupt:
    raise SystemExit(0)

excluded = sum(1 for job in tile_jobs if job["binary_mask_excluded"])
print(f"Excluded  : {excluded}/{len(tile_jobs)} background")

save_json(tile_jobs, ANNOTATION_PATH)

print(f"pairs     : {len(pairs)}")
print(f"tile jobs : {len(tile_jobs)}")
print(f"saved to  : {ANNOTATION_PATH}")

if TEST_RUN:
    print("WARNING: TEST_RUN overwrites annotation JSON with a single-pair subset.")

print("\nFirst job:")
print(tile_jobs[0])

print("\nLast job:")
print(tile_jobs[-1])

pair_id 0: HE 6045 / IHC 6036
  d0: grid 1x1, page 4, tile 2054x1388
  d1: grid 2x2, page 4, tile 1027x694
  d2: grid 4x4, page 4, tile 513x347
  d3: grid 8x8, page 3, tile 1027x694
  d4: grid 16x16, page 3, tile 513x347
